Squad 2 | Camada Bronze

**Tabela** | ecommerce_categorias |

**Origem** | vendas_raw/ (parquet) |

**Destino** | squad2/bronze/ecommerce_categorias (Delta) |

**Modo** | Delta Streaming — Structured Streaming |

**Objetivo** | Ingerir dados brutos na camada Bronze |

**Checkpoint** | squad2/checkpoints/bronze/ecommerce_categorias |

**Depende de** | feat_squad2_99_helpers |

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
import logging
from pyspark.sql.functions import lit, current_timestamp
from pyspark.sql.types import (
    StructType, StructField,
    LongType, StringType, DoubleType
)

logging.getLogger("azure").setLevel(logging.WARNING)

TABELA          = "ecommerce_categorias"
BASE_URL        = f"abfss://{ADLS_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net"
BRONZE_PATH     = f"{BASE_URL}/squad2/bronze/{TABELA}"
CHECKPOINT_PATH = f"{BASE_URL}/squad2/checkpoints/bronze/{TABELA}"

inicio = log_inicio(f"feat_squad2_bronze_{TABELA}")

log.info(f"Tabela         : {TABELA}")
log.info(f"Bronze Path    : {BRONZE_PATH}")
log.info(f"Checkpoint Path: {CHECKPOINT_PATH}")

In [0]:
SCHEMA = StructType([
    StructField("id_categoria",     LongType(),   True),
    StructField("nome_categoria",   StringType(), True),
    StructField("id_categoria_pai", DoubleType(), True)
])

log.info("Schema definido!")

In [0]:
try:
    snapshots = sorted(listar_snapshots())
    log.info(f"{len(snapshots)} snapshot(s) disponível(is):\n")
    for snap in snapshots:
        print(f"  Pacotes: {snap}")

except Exception as e:
    log.error(f"Erro ao listar snapshots: {str(e)}")
    raise

In [0]:
try:
    container_client    = get_container_client()
    checkpoint_file     = f"squad2/checkpoints/bronze/{TABELA}/processed_snapshots.txt"
    processados         = set()

    try:
        file_client  = container_client.get_file_client(checkpoint_file)
        download     = file_client.download_file()
        conteudo     = download.readall().decode("utf-8")
        processados  = set(conteudo.strip().split("\n")) if conteudo.strip() else set()
        log.info(f"{len(processados)} snapshot(s) já processado(s)")
    except Exception:
        log.info("Nenhum checkpoint encontrado — processando todos os snapshots")

    novos = [s for s in snapshots if s not in processados]
    log.info(f"{len(novos)} snapshot(s) novo(s) para processar")

except Exception as e:
    log.error(f"Erro ao verificar checkpoint: {str(e)}")
    raise

In [0]:
try:
    total_linhas = 0

    if not novos:
        log.info("Nenhum snapshot novo para processar!")
    else:
        for snapshot_id in novos:
            log.info(f"Processando: {snapshot_id}")

            # Lê via Azure SDK
            df = ler_parquet(snapshot_id, TABELA)

            # Adiciona colunas de controle
            df_bronze = df \
                .withColumn("_snapshot_id", lit(snapshot_id)) \
                .withColumn("_ingested_at", current_timestamp()) \
                .withColumn("_source",      lit("vendas_raw")) \
                .withColumn("_camada",      lit("bronze"))

            # Grava em Delta no ADLS
            df_bronze.write \
                .format("delta") \
                .mode("append") \
                .option("mergeSchema", "true") \
                .save(BRONZE_PATH)

            count         = df_bronze.count()
            total_linhas += count
            processados.add(snapshot_id)

            log.info(f"  ✅ {snapshot_id} → {count} linhas gravadas")

        # Atualiza checkpoint no ADLS via Azure SDK
        file_client = container_client.get_file_client(checkpoint_file)
        conteudo    = "\n".join(processados).encode("utf-8")

        try:
            file_client.get_file_properties()
            file_client.upload_data(conteudo, overwrite=True)
        except Exception:
            file_client.create_file()
            file_client.upload_data(conteudo, overwrite=True)

        log.info(f" Checkpoint atualizado → {len(processados)} snapshots")
        log.info(f" Total gravado na Bronze: {total_linhas} linhas")

except Exception as e:
    log.error(f"Erro na ingestão Bronze: {str(e)}")
    raise


In [0]:
try:
    df_bronze = spark.read.format("delta").load(BRONZE_PATH)
    total     = df_bronze.count()

    log.info(f"✅ Validação Bronze OK!")
    log.info(f"   Total registros : {total}")
    log.info(f"   Colunas         : {len(df_bronze.columns)}")

    print("\n📋 Schema Bronze:")
    df_bronze.printSchema()

    print("\n📊 Amostra:")
    display(df_bronze)

except Exception as e:
    log.error(f"Erro na validação: {str(e)}")
    raise


In [0]:
try:
    from delta.tables import DeltaTable

    delta_table = DeltaTable.forPath(spark, BRONZE_PATH)
    log.info("Histórico Delta:")
    display(delta_table.history())

except Exception as e:
    log.error(f"Erro ao verificar histórico: {str(e)}")

In [0]:
log_fim(f"feat_squad2_bronze_{TABELA}", inicio)